In [6]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.datasets import QM9
from torch_geometric.utils import to_dense_adj, to_dense_batch

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset dimensions
N_NODES = 9       # Max heavy atoms per molecule in this subset
N_ATOMS = 5       # Types: Empty (padding), C, N, O, F
N_BONDS = 5       # Types: None, Single, Double, Triple, Aromatic
SAMPLE_LIMIT = 5000

print(f"Loading QM9 on {device}...")
dataset = QM9(root="data/QM9")

node_tensors, adj_tensors = [], []

for data in dataset:
    if data.num_nodes > N_NODES:
        continue

    # Extract heavy atom features (indices 1 to 4 correspond to C, N, O, F)
    x_heavy = data.x[:, 1:5]
    x_dense, mask = to_dense_batch(x_heavy, max_num_nodes=N_NODES)
    x_dense = x_dense[0]

    # Prepend a channel indicating whether the node is an empty padding node
    is_empty = (~mask[0]).float().unsqueeze(1)
    x_final = torch.cat([is_empty, x_dense], dim=1)

    # Dense adjacency matrix
    adj_dense = to_dense_adj(data.edge_index, edge_attr=data.edge_attr, max_num_nodes=N_NODES)[0]
    has_bond = adj_dense.sum(dim=-1)
    no_bond = (has_bond == 0).float().unsqueeze(-1)
    no_bond.diagonal().fill_(1.0)  # Self-loops represent 'no bond'

    adj_final = torch.cat([no_bond, adj_dense], dim=-1)

    node_tensors.append(x_final)
    adj_tensors.append(adj_final)

    if len(node_tensors) >= SAMPLE_LIMIT:
        break

stacked_nodes = torch.stack(node_tensors)
stacked_adjs = torch.stack(adj_tensors)

dataloader = DataLoader(
    TensorDataset(stacked_nodes, stacked_adjs),
    batch_size=32,
    shuffle=True
)

print(f"Loaded {len(stacked_nodes)} molecules.")
print(f"Node tensor shape: {stacked_nodes.shape} | Adj tensor shape: {stacked_adjs.shape}")

Loading QM9 on cpu...
Loaded 280 molecules.
Node tensor shape: torch.Size([280, 9, 5]) | Adj tensor shape: torch.Size([280, 9, 9, 5])


Step 1: Environment Setup & QM9 Molecular Graph Extraction
Theoretical Overview
In MolGAN-style architectures, 
molecules are modeled as discrete annotated graphs $G = (V, E)$ :

- Node Matrix ($X$): Represents atom species ($C, N, O, F$) alongside an explicit empty padding token. Dimension: $\mathbb{R}^{B \times N_{\text{nodes}} \times N_{\text{atoms}}}$.
- Adjacency Tensor ($A$): Represents bond orders (None, Single, Double, Triple, Aromatic). Dimension: $\mathbb{R}^{B \times N_{\text{nodes}} \times N_{\text{nodes}} \times N_{\text{bonds}}}$.

Unlike sparse message-passing approaches, the dense adversarial generator outputs complete continuous probability matrices over the entire molecular adjacency structure.


Step 2: Dimension Reduction via PCA for Photonic Circuit Encoding
Theoretical Overview
Continuous-variable (CV) quantum circuits represent quantum states in optical qumodes through displacement operators $\hat{D}(\alpha)$ and squeezing operators $\hat{S}(r, \theta)$.Because a flattened molecular graph contains $9 \times 9 \times 5 + 9 \times 5 = 450$ dimensions, feeding it directly into a parameterized quantum circuit (PQC) is intractable. We fit an empirical Principal Component Analysis (PCA) projection matrix:
$$\Phi(x) = W_{\text{PCA}} (x - \mu)$$


This projects the classical feature vector onto $M = 6$ principal coordinates matching the physical optical modes.

In [7]:
from sklearn.decomposition import PCA

NB_MODES = 6  # Number of qumodes in the photonic circuit

# Flatten graph adjacency and node matrices to fit PCA
sample_adj = stacked_adjs.flatten(start_dim=1).cpu().numpy()
sample_nodes = stacked_nodes.flatten(start_dim=1).cpu().numpy()
sample_features = np.concatenate([sample_adj, sample_nodes], axis=1)

print(f"Fitting PCA from {sample_features.shape[1]} features down to {NB_MODES} modes...")
pca = PCA(n_components=NB_MODES)
pca.fit(sample_features)

print("Explained variance ratio:", pca.explained_variance_ratio_)

Fitting PCA from 450 features down to 6 modes...
Explained variance ratio: [0.10263668 0.0921095  0.07063147 0.06424611 0.05117226 0.04340701]


Step 3: Neural & Quantum Module Initialization
Theoretical Overview
The architecture relies on a tri-network system:
1. Generator ($G_{\theta}$): An MLP processing latent vector $z \sim \mathcal{N}(0, I)$ into logit matrices for bonds and atoms.
2. Critic / Discriminator ($D_{\phi}$): Relational Graph Convolutional Network (RGCN) estimating the 1-Wasserstein distance with Gradient Penalty (WGAN-GP).
3. Surrogate Quantum Reward Predictor ($Q_{\psi}$): A variational photonic quantum layer that maps molecular inputs to an estimated chemical reward $R_Q$.

In [8]:
from lib.WGAN import Generator, Discriminator, gradient_penalty
from lib.quantum_layer import PhotonicRewardModule
from lib.chemical_reward import evaluate_and_reward

Z_DIM = 8
G_CONV_DIMS = [128, 256, 512]
D_CONV_DIMS = ([128, 64], 128, [128, 64])

# Initialize models
G = Generator(G_CONV_DIMS, Z_DIM, N_NODES, N_BONDS, N_ATOMS, dropout_rate=0.0).to(device)
D = Discriminator(D_CONV_DIMS, m_dim=N_ATOMS, b_dim=N_BONDS - 1, dropout_rate=0.0).to(device)

Q = PhotonicRewardModule(
    pca_components=pca.components_,
    pca_mean=pca.mean_,
    num_atoms=N_NODES,
    num_bonds=N_BONDS,
    num_atom_types=N_ATOMS,
    nb_modes=NB_MODES
).to(device)

# Optimizers
opt_g = torch.optim.RMSprop(G.parameters(), lr=1e-4)
opt_d = torch.optim.RMSprop(D.parameters(), lr=1e-4)
opt_q = torch.optim.RMSprop(Q.parameters(), lr=1e-4)

Step 4: The 3-Step Hybrid Adversarial-RL Loop
Theoretical Overview
The training loop alternates between three objectives:

- Step A (WGAN-GP Critic): Updates $D_{\phi}$ on discrete graphs ($hard = \text{True}$) sampled via Gumbel-Softmax:
$$\mathcal{L}_D = -\mathbb{E}[D(X_{\text{real}})] + \mathbb{E}[D(X_{\text{fake}})] + \lambda_{\text{gp}} \mathbb{E}[(\Vert{}\nabla_{\hat{X}} D(\hat{X})\Vert{}_2 - 1)^2]$$

- Step B (Photonic Reward Surrogate): Optimizes $Q_{\psi}$ via an $L_1$ loss to approximate the true non-differentiable RDKit chemical reward $R_C$ on both real and generated molecules:
$$\mathcal{L}_Q = \mathbb{E}[\vert{}Q(X_{\text{real}}) - R_C(X_{\text{real}})\vert{}] + \mathbb{E}[\vert{}Q(X_{\text{fake}}) - R_C(X_{\text{fake}})\vert{}]$$

- Step C (Generator Relaxation): Generator backpropagation uses continuous soft relaxations ($hard = \text{False}$) to enable gradient flow from both the discriminator and the quantum surrogate:
$$\mathcal{L}_G = (1 - \lambda)\mathcal{L}_{\text{WGAN}} - \lambda \mathbb{E}[R_Q]$$

$\lambda$ follows a scheduled warmup to prevent early policy collapse.

In [9]:
import time

EPOCHS = 10           # Set to 200 for full convergence[cite: 1]
N_CRITIC = 5          # Discriminator steps per generator step[cite: 1]
LAMBDA_GP = 10.0      # WGAN-GP coefficient[cite: 1]
TAU = 1.0             # Gumbel-Softmax temperature[cite: 1]
WARMUP_EPOCHS = 2     # Initial phase without RL[cite: 1]
RAMP_EPOCHS = 5       # Linear transition phase[cite: 1]
LAMBDA_MAX = 0.5      # Maximum RL weighting factor[cite: 1]
EMA_BETA = 0.9        # Smoothing factor for quantum reward tracking[cite: 1]

reward_ema = 0.0
global_step = 0

print("--- Starting MolGAN-QRL Training ---")

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    # Dynamic RL weighting: warmup -> linear ramp -> plateau
    if epoch <= WARMUP_EPOCHS:
        current_lambda = 0.0
    else:
        progress = (epoch - WARMUP_EPOCHS) / RAMP_EPOCHS
        current_lambda = min(LAMBDA_MAX, progress * LAMBDA_MAX)

    for batch_nodes, batch_adj in dataloader:
        batch_size = batch_nodes.size(0)
        batch_nodes, batch_adj = batch_nodes.to(device), batch_adj.to(device)

        # -----------------------------------------------------------------
        # STEP A: TRAIN DISCRIMINATOR (WGAN-GP)
        # -----------------------------------------------------------------
        for _ in range(N_CRITIC):
            opt_d.zero_grad()
            z = torch.randn(batch_size, Z_DIM, device=device)

            with torch.no_grad():
                edges_logits, nodes_logits = G(z)
                # Hard categorical sampling for realistic discriminator evaluation
                fake_adj = F.gumbel_softmax(edges_logits, tau=TAU, hard=True, dim=-1)
                fake_nodes = F.gumbel_softmax(nodes_logits, tau=TAU, hard=True, dim=-1)

            real_val = D(batch_adj, None, batch_nodes)
            fake_val = D(fake_adj, None, fake_nodes)

            gp = gradient_penalty(D, batch_nodes, batch_adj, fake_nodes, fake_adj, device)
            d_loss = -torch.mean(real_val) + torch.mean(fake_val) + (LAMBDA_GP * gp)

            d_loss.backward()
            opt_d.step()

        # -----------------------------------------------------------------
        # STEP B: TRAIN PHOTONIC SURROGATE REWARD MODULE (Equation 5)[cite: 1]
        # -----------------------------------------------------------------
        opt_q.zero_grad()
        with torch.no_grad():
            z = torch.randn(batch_size, Z_DIM, device=device)
            edges_logits, nodes_logits = G(z)
            fake_adj_hard = F.gumbel_softmax(edges_logits, tau=TAU, hard=True, dim=-1)
            fake_nodes_hard = F.gumbel_softmax(nodes_logits, tau=TAU, hard=True, dim=-1)

            # Evaluate real chemical properties using RDKit
            target_rc_fake, val_ratio, uniq_ratio = evaluate_and_reward(fake_adj_hard, fake_nodes_hard)
            target_rc_real, _, _ = evaluate_and_reward(batch_adj, batch_nodes)

        # Predict surrogate reward using the Photonic Module
        pred_rq_fake = Q(fake_adj_hard, fake_nodes_hard)
        pred_rq_real = Q(batch_adj, batch_nodes)

        # L1 regression loss between quantum predictions and RDKit ground truth
        q_loss = torch.mean(
            torch.abs(pred_rq_real - target_rc_real) + torch.abs(pred_rq_fake - target_rc_fake)
        )
        q_loss.backward()
        opt_q.step()

        # -----------------------------------------------------------------
        # STEP C: TRAIN GENERATOR (Continuous Relaxation & RL)
        # -----------------------------------------------------------------
        opt_g.zero_grad()
        z = torch.randn(batch_size, Z_DIM, device=device)
        edges_logits, nodes_logits = G(z)

        # Soft continuous relaxation allows end-to-end backpropagation through G
        fake_adj_soft = F.gumbel_softmax(edges_logits, tau=TAU, hard=False, dim=-1)
        fake_nodes_soft = F.gumbel_softmax(nodes_logits, tau=TAU, hard=False, dim=-1)

        fake_val = D(fake_adj_soft, None, fake_nodes_soft)
        g_loss_wgan = -torch.mean(fake_val)

        # Quantum reward tensor retains computation graph
        pred_rq_soft = Q(fake_adj_soft, fake_nodes_soft)
        instant_reward = torch.mean(pred_rq_soft)

        # Update EMA for tracking
        reward_ema = instant_reward.item() if global_step == 0 else EMA_BETA * reward_ema + (1.0 - EMA_BETA) * instant_reward.item()

        # Convex combination of WGAN loss and Reinforcement Learning loss[cite: 1]
        g_loss_total = (1.0 - current_lambda) * g_loss_wgan + current_lambda * (-instant_reward)

        g_loss_total.backward()
        opt_g.step()

        global_step += 1

    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"D_Loss: {d_loss.item():.4f} | "
        f"G_Loss: {g_loss_total.item():.4f} | "
        f"Validity: {val_ratio * 100:.1f}% | "
        f"Uniqueness: {uniq_ratio * 100:.1f}% | "
        f"Lambda: {current_lambda:.2f} | "
        f"Time: {elapsed:.2f}s"
    )

--- Starting MolGAN-QRL Training ---


Epoch 01/10 | D_Loss: -1.8885 | G_Loss: 0.9249 | Validity: 0.0% | Uniqueness: 0.0% | Lambda: 0.00 | Time: 6.62s
Epoch 02/10 | D_Loss: -0.4404 | G_Loss: -0.3387 | Validity: 0.0% | Uniqueness: 0.0% | Lambda: 0.00 | Time: 5.74s
Epoch 03/10 | D_Loss: -0.5518 | G_Loss: -1.1940 | Validity: 41.7% | Uniqueness: 100.0% | Lambda: 0.10 | Time: 4.85s
Epoch 04/10 | D_Loss: -0.6019 | G_Loss: -0.8459 | Validity: 4.2% | Uniqueness: 100.0% | Lambda: 0.20 | Time: 5.29s
Epoch 05/10 | D_Loss: -0.9747 | G_Loss: -0.6145 | Validity: 4.2% | Uniqueness: 100.0% | Lambda: 0.30 | Time: 6.09s
Epoch 06/10 | D_Loss: -1.1492 | G_Loss: -0.5303 | Validity: 16.7% | Uniqueness: 100.0% | Lambda: 0.40 | Time: 4.81s
Epoch 07/10 | D_Loss: -1.2505 | G_Loss: -0.5199 | Validity: 20.8% | Uniqueness: 100.0% | Lambda: 0.50 | Time: 5.85s
Epoch 08/10 | D_Loss: -1.2161 | G_Loss: -0.4608 | Validity: 16.7% | Uniqueness: 100.0% | Lambda: 0.50 | Time: 4.58s
Epoch 09/10 | D_Loss: -1.1979 | G_Loss: -0.5326 | Validity: 12.5% | Uniqueness: 1

Step 5: Sampling and Chemical Quality Evaluation
Theoretical Overview
During testing and inference:
1. $G_{\theta}$ is set to evaluation mode.
2. We feed standard normal samples $z \sim \mathcal{N}(0, I)$ and apply hard argmax projection to obtain valid adjacency graphs.
3. RDKit parses the generated bond and atom valences to evaluate key metrics:
    - Validity: Percentage of chemically valid graphs satisfying valence rules.
    - Uniqueness: Ratio of non-duplicated valid structures within the generated set.
    - NVU: the number of valid and unique molecules generated.

This ensure we can use this model to explore and generate several valid molecular structures, and try to combine them to try to discover new drugs.

In [10]:
# Inference mode
G.eval()
num_test_samples = 100

with torch.no_grad():
    z = torch.randn(num_test_samples, Z_DIM, device=device)
    edges_logits, nodes_logits = G(z)

    # Argmax decoding
    adj_test = F.gumbel_softmax(edges_logits, tau=TAU, hard=True, dim=-1)
    nodes_test = F.gumbel_softmax(nodes_logits, tau=TAU, hard=True, dim=-1)

    rewards, val_score, uniq_score = evaluate_and_reward(adj_test, nodes_test)

print(f"--- Final Evaluation on {num_test_samples} Samples ---")
print(f"Validity Rate   : {val_score * 100:.2f}%")
print(f"Uniqueness Rate : {uniq_score * 100:.2f}%")
print(f"Mean Score      : {rewards.mean().item():.4f}")

--- Final Evaluation on 100 Samples ---
Validity Rate   : 16.00%
Uniqueness Rate : 100.00%
Mean Score      : 0.0424
